<a href="https://colab.research.google.com/github/azrapatvi/dl-practice/blob/main/5_customer_churn_prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("rjmanoj/credit-card-customer-churn-prediction")

print("Path to dataset files:", path)

100%|██████████| 262k/262k [00:00<00:00, 384kB/s]

Extracting files...
Path to dataset files: /root/.cache/kagglehub/datasets/rjmanoj/credit-card-customer-churn-prediction/versions/1


In [ ]:
import os

df=pd.read_csv(os.path.join(path,'Churn_Modelling.csv'))

In [ ]:
df.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [ ]:
df.shape

(10000, 14)

In [ ]:
df.drop(['RowNumber','CustomerId','Surname'],axis=1,inplace=True)

In [ ]:
df.columns=df.columns.str.lower()

In [ ]:
df.columns

Index(['creditscore', 'geography', 'gender', 'age', 'tenure', 'balance',
       'numofproducts', 'hascrcard', 'isactivemember', 'estimatedsalary',
       'exited'],
      dtype='object')

In [ ]:
df.isnull().sum()

,0
creditscore,0
geography,0
gender,0
age,0
tenure,0
balance,0
numofproducts,0
hascrcard,0
isactivemember,0
estimatedsalary,0


In [ ]:
df.duplicated().sum()

np.int64(0)

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   creditscore      10000 non-null  int64  
 1   geography        10000 non-null  object 
 2   gender           10000 non-null  object 
 3   age              10000 non-null  int64  
 4   tenure           10000 non-null  int64  
 5   balance          10000 non-null  float64
 6   numofproducts    10000 non-null  int64  
 7   hascrcard        10000 non-null  int64  
 8   isactivemember   10000 non-null  int64  
 9   estimatedsalary  10000 non-null  float64
 10  exited           10000 non-null  int64  
dtypes: float64(2), int64(7), object(2)
memory usage: 859.5+ KB


In [ ]:
df['exited'].value_counts()

,count
exited,
0,7963
1,2037


In [ ]:
X=df.drop('exited',axis=1)
y=df['exited']

In [ ]:
num_cols=[i for i in X.columns if X[i].dtype!='O']
num_cols

['creditscore',
 'age',
 'tenure',
 'balance',
 'numofproducts',
 'hascrcard',
 'isactivemember',
 'estimatedsalary']

In [ ]:
cat_cols=[i for i in X.columns if X[i].dtype=='O']
cat_cols

['geography', 'gender']

In [ ]:
from sklearn.preprocessing import StandardScaler,OneHotEncoder
from sklearn.compose import ColumnTransformer

scaler=StandardScaler()
ohe=OneHotEncoder(handle_unknown='ignore',drop='first')

preprocessor=ColumnTransformer([
    ("scaler",scaler,num_cols),
    ("ohe",ohe,cat_cols)
])

preprocessor

ColumnTransformer(transformers=[('scaler', StandardScaler(),
                                 ['creditscore', 'age', 'tenure', 'balance',
                                  'numofproducts', 'hascrcard',
                                  'isactivemember', 'estimatedsalary']),
                                ('ohe',
                                 OneHotEncoder(drop='first',
                                               handle_unknown='ignore'),
                                 ['geography', 'gender'])])

In [ ]:
from sklearn.model_selection import train_test_split

X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42,stratify=y)

In [ ]:
X_train_scaled=preprocessor.fit_transform(X_train)
X_test_scaled=preprocessor.transform(X_test)

In [ ]:
X_train_scaled.shape[1],


(11,)

In [ ]:
y_train

,exited
2151,1
8392,1
5006,0
4117,0
7182,0
...,...
4555,1
4644,0
8942,0
2935,0


In [ ]:
from sklearn.utils.class_weight import compute_sample_weight
import numpy as np

class_weight = compute_sample_weight(
    class_weight='balanced',
    y=y_train
)

In [ ]:
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense

model=Sequential([
    Dense(16,activation='relu',input_dim=X_train_scaled.shape[1],),
    Dense(8,activation='relu'),
    Dense(8,activation='relu'),
    Dense(1,activation='sigmoid')
])

/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [ ]:
model.summary()

Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_11 (Dense)                │ (None, 16)             │           192 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_12 (Dense)                │ (None, 8)              │           136 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_13 (Dense)                │ (None, 8)              │            72 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_14 (Dense)                │ (None, 1)              │             9 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 409 (1.60 KB)

 Trainable params: 409 (1.60 KB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
from tensorflow.keras.metrics import Precision, Recall

model.compile(optimizer='adam',loss='binary_crossentropy', metrics=[
        'accuracy',
        Precision(name='precision'),
        Recall(name='recall')
    ])

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

history = model.fit(
    X_train_scaled,
    y_train,
    epochs=50,
    batch_size=8,
    validation_split=0.2,
    sample_weight=weights,
    callbacks=[early_stop]
)

Epoch 1/50
800/800 ━━━━━━━━━━━━━━━━━━━━ 6s 5ms/step - accuracy: 0.6419 - loss: 0.6425 - precision: 0.3079 - recall: 0.6008 - val_accuracy: 0.6925 - val_loss: 0.5726 - val_precision: 0.3673 - val_recall: 0.7437
Epoch 2/50
800/800 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.7289 - loss: 0.5359 - precision: 0.4096 - recall: 0.7351 - val_accuracy: 0.7563 - val_loss: 0.5077 - val_precision: 0.4352 - val_recall: 0.7344
Epoch 3/50
800/800 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.7563 - loss: 0.4987 - precision: 0.4434 - recall: 0.7473 - val_accuracy: 0.7331 - val_loss: 0.4897 - val_precision: 0.4147 - val_recall: 0.8125
Epoch 4/50
800/800 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.7697 - loss: 0.4836 - precision: 0.4621 - recall: 0.7634 - val_accuracy: 0.7781 - val_loss: 0.4740 - val_precision: 0.4665 - val_recall: 0.7625
Epoch 5/50
800/800 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.7758 - loss: 0.4750 - precision: 0.4703 - recall: 0.7542 - val_accuracy: 0.7925 - val_loss: 0.

In [ ]:
y_pred=model.predict(X_test_scaled)

y_pred

63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step


array([[0.08894385],
       [0.19301108],
       [0.19364543],
       ...,
       [0.962503  ],
       [0.07560408],
       [0.37183604]], dtype=float32)

In [ ]:
loss, acc, precision, recall = model.evaluate(
    X_test_scaled,
    y_test
)

print("Loss:", loss)
print("Accuracy:", acc)
print("Precision:", precision)
print("Recall:", recall)

63/63 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.7650 - loss: 0.4841 - precision: 0.4549 - recall: 0.7813
Loss: 0.48410433530807495
Accuracy: 0.7649999856948853
Precision: 0.45493561029434204
Recall: 0.7813267707824707


In [ ]:
new_customer = pd.DataFrame({
    'creditscore': [620],
    'geography': ['Germany'],
    'gender': ['Female'],
    'age': [45],
    'tenure': [3],
    'balance': [125000],
    'numofproducts': [1],
    'hascrcard': [1],
    'isactivemember': [0],
    'estimatedsalary': [85000]
})

new_customer

,creditscore,geography,gender,age,tenure,balance,numofproducts,hascrcard,isactivemember,estimatedsalary
0,620,Germany,Female,45,3,125000,1,1,0,85000


In [ ]:
new_customer_scaled=preprocessor.transform(new_customer)

In [ ]:
new_customer_scaled

array([[-0.31838052,  0.57507594, -0.6962018 ,  0.78042101, -0.91025649,
         0.64104192, -1.030206  , -0.25694083,  1.        ,  0.        ,
         0.        ]])

In [ ]:
new_pred = model.predict(new_customer_scaled)

new_class = (new_pred >= 0.5).astype(int)

print("Churn Probability:", new_pred[0][0])
print("Prediction:", new_class[0][0])

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 85ms/step
Churn Probability: 0.8755792
Prediction: 1


In [ ]:
df.head(2)

,creditscore,geography,gender,age,tenure,balance,numofproducts,hascrcard,isactivemember,estimatedsalary,exited
0,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
